# Human rating analysis (Project 23)

This notebook reads the 3 rater files, matches them to Lead-3 / BART using the answer key, and calculates the averages, significance test and rater agreement.

## 1. Load files

In [3]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import wilcoxon, spearmanr

rater_files = ["rater1.xlsx", "rater2.xlsx", "rater3.xlsx"]
cols = ["relevance", "fluency", "accuracy"]

key = pd.read_csv("answer_key.csv")         
raters = [pd.read_excel(f).sort_values("row_id").reset_index(drop=True) for f in rater_files]

## 2. Check the files

In [4]:
for name, d in zip(rater_files, raters):
    missing = d[cols].isna().sum().sum()
    bad = (~d[cols].isin([1, 2, 3, 4, 5])).sum().sum()
    print(f"{name}: rows={len(d)}, missing={missing}, values outside 1-5={bad}")

rater1.xlsx: rows=40, missing=0, values outside 1-5=0
rater2.xlsx: rows=40, missing=0, values outside 1-5=0
rater3.xlsx: rows=40, missing=0, values outside 1-5=0


## 3. Average rating per method

In [5]:
# Put all raters in one table and add the method (lead3 / bart) from the answer key
frames = []
for i, d in enumerate(raters, start=1):
    x = d.merge(key, on="row_id")
    x["rater"] = i
    frames.append(x)
ratings = pd.concat(frames, ignore_index=True)

# Average rating per method (all raters together)
summary = ratings.groupby("method")[cols].mean().round(2)
summary

,relevance,fluency,accuracy
method,,,
bart,4.45,4.93,4.43
lead3,4.63,4.93,4.58


In [7]:
# Average per rater and method
ratings.groupby(["rater", "method"])[cols].mean().round(2)

relevance  fluency  accuracy
rater method                              
1     bart         4.50     4.95      4.50
      lead3        4.65     4.95      4.60
2     bart         4.40     4.85      4.40
      lead3        4.65     4.90      4.55
3     bart         4.45     5.00      4.40
      lead3        4.60     4.95      4.60

## 4. Is the difference real?

In [8]:
# Article-by-article comparison: is BART really different from Lead-3?
per_article = ratings.groupby(["article_id", "method"])[cols].mean().reset_index()
wide = per_article.pivot(index="article_id", columns="method")

rows = []
for c in cols:
    diff = wide[(c, "bart")] - wide[(c, "lead3")]
    p = wilcoxon(diff).pvalue if (diff != 0).any() else np.nan
    rows.append({"measure": c,
                 "BART better": int((diff > 0).sum()),
                 "BART worse": int((diff < 0).sum()),
                 "tie": int((diff == 0).sum()),
                 "mean difference (BART - Lead-3)": round(diff.mean(), 2),
                 "Wilcoxon p-value": round(p, 3)})
pd.DataFrame(rows)
# If p-value is bigger than 0.05, the difference is not statistically significant.

,measure,BART better,BART worse,tie,mean difference (BART - Lead-3),Wilcoxon p-value
0,relevance,6,9,5,-0.18,0.241
1,fluency,2,4,14,0.00,0.739
2,accuracy,6,6,8,-0.15,0.723


## 5. Do raters agree?

In [9]:
# How much do raters agree? (exact same score, and rank correlation) for each pair of raters
rows = []
for c in cols:
    for a, b in combinations(range(3), 2):
        exact = (raters[a][c] == raters[b][c]).mean()
        within1 = ((raters[a][c] - raters[b][c]).abs() <= 1).mean()
        rho = spearmanr(raters[a][c], raters[b][c]).correlation
        rows.append({"measure": c, "pair": f"rater{a+1} vs rater{b+1}",
                     "exact agreement": round(exact, 2),
                     "within 1 point": round(within1, 2),
                     "Spearman": round(rho, 2)})
pd.DataFrame(rows)


,measure,pair,exact agreement,within 1 point,Spearman
0,relevance,rater1 vs rater2,0.72,0.98,0.63
1,relevance,rater1 vs rater3,0.75,1.00,0.68
2,relevance,rater2 vs rater3,0.72,0.98,0.69
3,fluency,rater1 vs rater2,0.88,1.00,0.26
4,fluency,rater1 vs rater3,0.98,1.00,0.70
5,fluency,rater2 vs rater3,0.90,1.00,0.42
6,accuracy,rater1 vs rater2,0.92,1.00,0.88
7,accuracy,rater1 vs rater3,0.80,1.00,0.82
8,accuracy,rater2 vs rater3,0.72,1.00,0.72


## 6. Lowest-accuracy summaries

In [10]:
# Lowest-accuracy summaries 
low = ratings.groupby(["article_id", "method"])[cols].mean().reset_index()
low.sort_values("accuracy").head(5)

,article_id,method,relevance,fluency,accuracy
12,15,bart,3.666667,5.0,1.666667
36,47,bart,4.000000,5.0,2.666667
21,26,lead3,3.333333,5.0,3.000000
7,8,lead3,2.666667,5.0,3.000000
23,30,lead3,4.000000,5.0,3.666667


## 7. Save results

In [11]:
summary.to_csv("human_rating_results.csv")
print("Saved human_rating_results.csv")

Saved human_rating_results.csv
